<table style="width: 100%; border: none;">
    <tr style="border: none;">
        <td style="width: 30%; border: none; text-align: left;">
            <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/a/a0/Logo_de_la_UPTC.svg/960px-Logo_de_la_UPTC.svg.png" alt="Logo UPTC" width="250"/>
        </td>
        <td style="width: 70%; border: none; text-align: center;">
            <h3 style="margin: 0; padding: 2px;">Especialización en Analítica Estratégica de Datos</h3>
            <h4 style="margin: 0; padding: 2px; font-weight: normal;">Estadística para Analítica de Datos</h4>
            <h4 style="margin: 0; padding: 2px; font-weight: normal;">Proyecto Final - Modelación</h4>
            <h3 style="margin: 0; padding: 5px;">Entregable # 3</h3>
        </td>
    </tr>
</table>

**Profesor:** Duván Cataño

**Estudiantes:**

- Gabriela VIllalobos
- Marco Diaz
- Ivan Mariño
- Ivan Corredor
- Diego Reyes

# 1. Introducción y Objetivo

La gestión eficiente de los recursos destinadas a comisiones y viaticos, es un aspecto crucial para la optimización de recursos y la transparencia en las instituciones públicas como la Registraduría Nacional del Estado Civil. El presente trabajo aborda el análisis de un registro histórico de comisiones de viaje, centrando su enfoque en el proposito de viaje y el gasto realizado en cada uno de estos. La "limpieza" del dataset seleccionado, representa un análisis exploratorio y a partir de la aplicación de pruebas de hipotesis estadísticos para el desarrollo de modelos predictivos, se logra identificar patrones de comportamiento administrativo y gasto de viaticos, que permitirán evaluar la eficiencia de los recursos y sustentarán en datos la futuras decisiones de la entidad.

## 1.1 Objetivo

Analizar los determinantes del gasto en viáticos y comisiones institucionales mediante el uso de inferencia estadística y modelos de machine learning predictivos, con el fin de identificar patrones de consumo, auditar la distribución del presupuesto y generar herramientas que optimicen la planeación financiera de la entidad.

# 2. Descripción de variables

<table border="1" cellspacing="0" cellpadding="6">
    <thead>
        <tr>
            <th>Nombre</th>
            <th>Tipo</th>
            <th>Tipo de variable</th>
            <th>Descripción</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>viatico_id</td>
            <td>int</td>
            <td>Identificador</td>
            <td>Un identificador único para cada registro de gastos de viaje o «viático».</td>
        </tr>
        <tr>
            <td>commission_request_id</td>
            <td>int</td>
            <td>Identificador</td>
            <td>Un identificador único para la solicitud de comisión o reembolso asociada al gasto de viaje.</td>
        </tr>
        <tr>
            <td>total_travel_allowance</td>
            <td>float</td>
            <td>Cuantitativa continua</td>
            <td>El importe total de la asignación de viaje o del reembolso del gasto.</td>
        </tr>
        <tr>
            <td>purpose</td>
            <td>string</td>
            <td>Cualitativa nominal</td>
            <td>La categoría o clasificación del motivo o razón del gasto de viaje.</td>
        </tr>
        <tr>
            <td>full_date</td>
            <td>date</td>
            <td>Cualitativa ordinal</td>
            <td>La fecha completa en el formato AAAA-MM-DD de inicio del viaje.</td>
        </tr>
        <tr>
            <td>commission_days</td>
            <td>int</td>
            <td>Cuantitativa discreta</td>
            <td>El número de días durante los cuales se incurrió en el gasto de viaje.</td>
        </tr>
        <tr>
            <td>employee_position_norm</td>
            <td>string</td>
            <td>Cualitativa nominal</td>
            <td>La versión normalizada y estandarizada del cargo o puesto del empleado.</td>
        </tr>
        <tr>
            <td>position_level</td>
            <td>int</td>
            <td>Cualitativa ordinal</td>
            <td>Un indicador del nivel jerárquico del empleado dentro de la organización.</td>
        </tr>
        <tr>
            <td>city_origin</td>
            <td>string</td>
            <td>Cualitativa nominal</td>
            <td>La ciudad de origen de la ruta de transporte.</td>
        </tr>
        <tr>
            <td>city_destination_main</td>
            <td>string</td>
            <td>Cualitativa nominal</td>
            <td>La ciudad de destino principal de la ruta de transporte.</td>
        </tr>
        <tr>
            <td>distance_km</td>
            <td>float</td>
            <td>Cuantitativa continua</td>
            <td>Distancia en kilómetros entre la ciudad de origen y destino.</td>
        </tr>
        <tr>
            <td>destination_count</td>
            <td>int</td>
            <td>Cuantitativa discreta</td>
            <td>El número de ciudades de destino a lo largo de la ruta.</td>
        </tr>
        <tr>
            <td>is_contractor</td>
            <td>boolean</td>
            <td>Cualitativa nominal dicotómica</td>
            <td>Un indicador que señala si el empleado es un contratista o un empleado a tiempo completo.</td>
        </tr>
    </tbody>
</table>


# 3. Limpieza y Preprocesamiento

In [ ]:
# @title 3.1. Creación de Funciones

def contar_outliers_iqr(df, columna, multiplicador=1.5):
    """
    Cuenta outliers usando el método IQR.
    """

    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)

    IQR = Q3 - Q1

    limite_inferior = Q1 - (multiplicador * IQR)
    limite_superior = Q3 + (multiplicador * IQR)

    outliers = df[
        (df[columna] < limite_inferior) |
        (df[columna] > limite_superior)
    ]

    return len(outliers)

def tabla_frecuencias_categorica(
    df,
    columna,
    ordenar='index',
    labels=None,
    mostrar_acumuladas=False,
    mostrar_tabla=True,
    width=1150,
    height=450
):
    """
    Genera una tabla de frecuencias para una variable categórica.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame de entrada.

    columna : str
        Nombre de la columna categórica.

    ordenar : str
        'index' -> ordena por índice/categoría
        'freq'  -> ordena por frecuencia descendente

    labels : dict
        Diccionario con etiquetas descriptivas.

    mostrar_acumuladas : bool
        Mostrar frecuencias acumuladas.

    mostrar_tabla : bool
        Mostrar tabla interactiva.

    width : int
        Ancho figura.

    height : int
        Alto figura.

    Retorna
    -------
    pandas.DataFrame
    """

    # =========================================
    # Frecuencias absolutas
    # =========================================
    frecuencias = df[columna].value_counts(
        dropna=False
    )

    # =========================================
    # Ordenamiento
    # =========================================
    if ordenar == 'index':

        frecuencias = frecuencias.sort_index()

    elif ordenar == 'freq':

        frecuencias = frecuencias.sort_values(
            ascending=False
        )

    # =========================================
    # Construcción DataFrame
    # =========================================
    tabla = frecuencias.reset_index()

    tabla.columns = [
        columna,
        'Frecuencia'
    ]

    # =========================================
    # Etiquetas descriptivas
    # =========================================
    if labels is not None:

        tabla[f'{columna}_label'] = (
            tabla[columna]
            .map(labels)
            .fillna('Sin etiqueta')
        )

    # =========================================
    # Frecuencia relativa
    # =========================================
    tabla['Frecuencia Relativa'] = (
        tabla['Frecuencia']
        / tabla['Frecuencia'].sum()
    )

    # =========================================
    # Porcentaje
    # =========================================
    tabla['Porcentaje (%)'] = (
        tabla['Frecuencia Relativa'] * 100
    )

    # =========================================
    # Frecuencias acumuladas
    # =========================================
    if mostrar_acumuladas:

        tabla['Frecuencia Acumulada'] = (
            tabla['Frecuencia']
            .cumsum()
        )

        tabla['Frecuencia Relativa Acum.'] = (
            tabla['Frecuencia Relativa']
            .cumsum()
        )

        tabla['Porcentaje Acum. (%)'] = (
            tabla['Porcentaje (%)']
            .cumsum()
        )

    # =========================================
    # Redondeos
    # =========================================
    tabla['Frecuencia Relativa'] = (
        tabla['Frecuencia Relativa']
        .round(4)
    )

    tabla['Porcentaje (%)'] = (
        tabla['Porcentaje (%)']
        .round(2)
    )

    if mostrar_acumuladas:

        tabla['Frecuencia Relativa Acum.'] = (
            tabla['Frecuencia Relativa Acum.']
            .round(4)
        )

        tabla['Porcentaje Acum. (%)'] = (
            tabla['Porcentaje Acum. (%)']
            .round(2)
        )

    # =========================================
    # Reordenar columnas
    # =========================================
    columnas_finales = [columna]

    if labels is not None:

        columnas_finales.append(
            f'{columna}_label'
        )

    columnas_finales.extend([
        'Frecuencia',
        'Frecuencia Relativa',
        'Porcentaje (%)'
    ])

    if mostrar_acumuladas:

        columnas_finales.extend([
            'Frecuencia Acumulada',
            'Frecuencia Relativa Acum.',
            'Porcentaje Acum. (%)'
        ])

    tabla = tabla[columnas_finales]

    # =========================================
    # Mostrar tabla
    # =========================================
    if mostrar_tabla:

        fig = go.Figure(

            data=[

                go.Table(

                    header=dict(

                        values=[
                            f"<b>{col}</b>"
                            for col in tabla.columns
                        ],

                        fill_color='#1E3A8A',

                        font=dict(
                            color='white',
                            size=14
                        ),

                        align='center',

                        height=40
                    ),

                    cells=dict(

                        values=[
                            tabla[col]
                            for col in tabla.columns
                        ],

                        fill_color=[
                            [
                                '#F8FAFC'
                                if i % 2 == 0
                                else '#E2E8F0'
                                for i in range(len(tabla))
                            ]
                        ],

                        align='center',

                        font=dict(
                            color='#111827',
                            size=13
                        ),

                        height=34
                    )
                )
            ]
        )

        fig.update_layout(

            title={

                'text': (
                    f'Tabla de Frecuencias - '
                    f'{columna}'
                ),

                'x': 0.5,

                'font': {
                    'size': 24
                }
            },

            width=width,

            height=height,

            margin=dict(
                l=20,
                r=20,
                t=70,
                b=20
            )
        )

        fig.show()

    return tabla


def grafico_barras_categorica(
    df,
    columna,
    labels=None,
    ordenar='index',
    mostrar_porcentaje=False,
    orientation='v',
    width=850,
    height=500
):
    """
    Genera un gráfico de barras para una variable categórica.

    Parámetros
    ----------
    df : pandas.DataFrame

    columna : str
        Variable categórica.

    labels : dict
        Diccionario de etiquetas descriptivas.

    ordenar : str
        'index' -> orden natural
        'freq'  -> orden por frecuencia

    mostrar_porcentaje : bool
        Si True muestra porcentajes.

    orientation : str
        'v' -> vertical
        'h' -> horizontal

    width : int
        Ancho figura.

    height : int
        Alto figura.
    """

    # =====================================
    # Frecuencias
    # =====================================
    frecuencias = (
        df[columna]
        .value_counts(dropna=False)
    )

    # =====================================
    # Ordenamiento
    # =====================================
    if ordenar == 'index':

        frecuencias = frecuencias.sort_index()

    elif ordenar == 'freq':

        frecuencias = frecuencias.sort_values(
            ascending=False
        )

    # =====================================
    # DataFrame base
    # =====================================
    plot_df = frecuencias.reset_index()

    plot_df.columns = [
        columna,
        'Frecuencia'
    ]

    # =====================================
    # Etiquetas descriptivas
    # =====================================
    if labels is not None:

        plot_df['Label'] = (
            plot_df[columna]
            .map(labels)
            .fillna('Sin etiqueta')
        )

    else:

        plot_df['Label'] = (
            plot_df[columna]
            .astype(str)
        )

    # =====================================
    # Porcentajes
    # =====================================
    total = plot_df['Frecuencia'].sum()

    plot_df['Porcentaje'] = (
        plot_df['Frecuencia']
        / total * 100
    ).round(2)

    # =====================================
    # Texto barras
    # =====================================
    if mostrar_porcentaje:

        plot_df['Texto'] = (
            plot_df['Frecuencia'].astype(str)
            + ' ('
            + plot_df['Porcentaje'].astype(str)
            + '%)'
        )

    else:

        plot_df['Texto'] = (
            plot_df['Frecuencia']
            .astype(str)
        )

    # =====================================
    # Gráfico vertical
    # =====================================
    if orientation == 'v':

        fig = px.bar(
            plot_df,
            x='Label',
            y='Frecuencia',
            color='Label',
            text='Texto'
        )

        fig.update_layout(
            xaxis_title='Categoría',
            yaxis_title='Frecuencia'
        )

    # =====================================
    # Gráfico horizontal
    # =====================================
    else:

        fig = px.bar(
            plot_df,
            y='Label',
            x='Frecuencia',
            color='Label',
            text='Texto',
            orientation='h'
        )

        fig.update_layout(
            xaxis_title='Frecuencia',
            yaxis_title='Categoría'
        )

    # =====================================
    # Ajustes visuales
    # =====================================
    fig.update_traces(
        textposition='outside'
    )

    fig.update_layout(

        template='plotly_white',

        title={
            'text': f'Distribución de {columna}',
            'x': 0.5
        },

        showlegend=False,

        width=width,
        height=height
    )

    fig.show()

    return plot_df

def grafico_pastel_categorica(
    df,
    columna,
    labels=None,
    ordenar='freq',
    donut=True,
    hole=0.35,
    pull=0.03,
    width=800,
    height=650
):
    """
    Genera un gráfico de pastel/donut para
    una variable categórica.

    Parámetros
    ----------
    df : pandas.DataFrame

    columna : str
        Variable categórica.

    labels : dict
        Diccionario de etiquetas descriptivas.

    ordenar : str
        'index' -> orden natural
        'freq'  -> orden por frecuencia

    donut : bool
        Si True crea gráfico donut.

    hole : float
        Tamaño del agujero interno.

    pull : float
        Separación de sectores.

    width : int
        Ancho figura.

    height : int
        Alto figura.
    """

    # =====================================
    # Frecuencias
    # =====================================
    frecuencias = (
        df[columna]
        .value_counts(dropna=False)
    )

    # =====================================
    # Ordenamiento
    # =====================================
    if ordenar == 'index':

        frecuencias = frecuencias.sort_index()

    elif ordenar == 'freq':

        frecuencias = frecuencias.sort_values(
            ascending=False
        )

    # =====================================
    # DataFrame base
    # =====================================
    plot_df = frecuencias.reset_index()

    plot_df.columns = [
        columna,
        'Frecuencia'
    ]

    # =====================================
    # Etiquetas descriptivas
    # =====================================
    if labels is not None:

        plot_df['Label'] = (
            plot_df[columna]
            .map(labels)
            .fillna('Sin etiqueta')
        )

    else:

        plot_df['Label'] = (
            plot_df[columna]
            .astype(str)
        )

    # =====================================
    # Gráfico pastel
    # =====================================
    fig = px.pie(

        plot_df,

        names='Label',

        values='Frecuencia',

        hole=hole if donut else 0
    )

    # =====================================
    # Ajustes sectores
    # =====================================
    fig.update_traces(

        textinfo='percent+label',

        pull=[
            pull
            for _ in range(len(plot_df))
        ],

        hovertemplate=
        '<b>Categoría</b>: %{label}<br>' +
        '<b>Frecuencia</b>: %{value}<br>' +
        '<b>Porcentaje</b>: %{percent}<extra></extra>'
    )

    # =====================================
    # Layout
    # =====================================
    fig.update_layout(

        template='plotly_white',

        title={
            'text': f'Distribución de {columna}',
            'x': 0.5,

            'font': {
                'size': 22
            }
        },

        width=width,
        height=height
    )

    fig.show()

    return plot_df

def tabla_frecuencias_continua(
    df,
    columna,
    bins=None,
    width=1300,
    height=500,
    mostrar_tabla=True
):
    """
    Tabla de frecuencias para variables continuas
    usando regla de Sturges.

    Parámetros
    ----------
    df : pandas.DataFrame

    columna : str
        Variable continua.

    bins : int
        Número de intervalos.
        Si None usa Sturges.

    mostrar_tabla : bool
        Mostrar tabla Plotly.

    Retorna
    -------
    pandas.DataFrame
    """

    # =====================================
    # Serie limpia
    # =====================================
    serie = df[columna].dropna()

    # =====================================
    # Tamaño muestra
    # =====================================
    n = len(serie)

    # =====================================
    # Número de clases
    # =====================================
    if bins is None:

        k = int(
            round(
                1 + 3.322 * np.log10(n)
            )
        )

    else:

        k = bins

    # =====================================
    # Valores extremos
    # =====================================
    minimo = serie.min()
    maximo = serie.max()

    # =====================================
    # Rango
    # =====================================
    rango = maximo - minimo

    # =====================================
    # Amplitud
    # =====================================
    amplitud = rango / k

    # =====================================
    # Límites intervalos
    # =====================================
    limites = np.arange(
        minimo,
        maximo + amplitud,
        amplitud
    )

    # =====================================
    # Intervalos
    # =====================================
    intervalos = pd.cut(
        serie,
        bins=limites,
        include_lowest=True
    )

    # =====================================
    # Frecuencias
    # =====================================
    frecuencias = (
        intervalos
        .value_counts()
        .sort_index()
    )

    # =====================================
    # Tabla base
    # =====================================
    tabla = frecuencias.reset_index()

    tabla.columns = [
        'Intervalo',
        'Frecuencia'
    ]

    # =====================================
    # Límites inferiores/superiores
    # =====================================
    tabla['Límite Inferior'] = (
        tabla['Intervalo']
        .apply(lambda x: x.left)
        .astype(float)
    )

    tabla['Límite Superior'] = (
        tabla['Intervalo']
        .apply(lambda x: x.right)
        .astype(float)
    )

    # =====================================
    # Marca de clase
    # =====================================
    tabla['Marca de Clase'] = (
        tabla['Límite Inferior']
        + tabla['Límite Superior']
    ) / 2

    # =====================================
    # Frecuencia relativa
    # =====================================
    tabla['Frecuencia Relativa'] = (
        tabla['Frecuencia'] / n
    )

    # =====================================
    # Porcentaje
    # =====================================
    tabla['Porcentaje (%)'] = (
        tabla['Frecuencia Relativa'] * 100
    )

    # =====================================
    # Acumuladas
    # =====================================
    tabla['Frecuencia Acumulada'] = (
        tabla['Frecuencia']
        .cumsum()
    )

    tabla['Frecuencia Relativa Acum.'] = (
        tabla['Frecuencia Relativa']
        .cumsum()
    )

    tabla['Porcentaje Acum. (%)'] = (
        tabla['Porcentaje (%)']
        .cumsum()
    )

    # =====================================
    # Redondeos
    # =====================================
    columnas_redondeo = [
        'Intervalo',
        'Marca de Clase',
        'Frecuencia Relativa',
        'Frecuencia Relativa Acum.',
        'Porcentaje (%)',
        'Porcentaje Acum. (%)'
    ]

    tabla[columnas_redondeo] = (
        tabla[columnas_redondeo]
        .round(4)
    )

    tabla['Intervalo'] = (
    tabla['Intervalo']
    .astype(str)
    ).replace('(', '').replace(']', '')

    # =====================================
    # Visualización
    # =====================================
    if mostrar_tabla:

        fig = go.Figure(

            data=[

                go.Table(

                    header=dict(

                        values=[
                            f"<b>{col}</b>"
                            for col in tabla.columns
                        ],

                        fill_color='#1E3A8A',

                        font=dict(
                            color='white',
                            size=13
                        ),

                        align='center',
                        height=40
                    ),

                    cells=dict(

                        values=[
                            tabla[col]
                            for col in tabla.columns
                        ],

                        fill_color=[
                            [
                                '#F8FAFC'
                                if i % 2 == 0
                                else '#E2E8F0'
                                for i in range(len(tabla))
                            ]
                        ],

                        align='center',

                        font=dict(
                            color='#111827',
                            size=12
                        ),

                        height=34
                    )
                )
            ]
        )

        fig.update_layout(

            title={
                'text': (
                    f'Tabla de Frecuencias '
                    f'Variable Continua - {columna}'
                ),

                'x': 0.5,

                'font': {
                    'size': 22
                }
            },

            width=width,
            height=height,

            margin=dict(
                l=20,
                r=20,
                t=70,
                b=20
            )
        )

        fig.show()

    return tabla


def histograma_variable_continua(
    df,
    columna,
    bins='sturges',
    marginal='box',
    mostrar_media=True,
    mostrar_mediana=True,
    mostrar_asimetria=True,
    mostrar_curtosis=True,
    width=950,
    height=600
):
    """
    Histograma analítico para variable continua.

    Incluye:
    - asimetría
    - curtosis
    - media
    - mediana
    - densidad
    - boxplot marginal

    Parámetros
    ----------
    df : pandas.DataFrame

    columna : str
        Variable continua.

    bins : int o str
        - 'sturges'
        - 'sqrt'
        - 'fd'

    marginal : str
        - 'box'
        - 'rug'
        - 'violin'
        - None
    """

    # =====================================
    # Serie limpia
    # =====================================
    valores = (
        df[columna]
        .dropna()
    )

    # =====================================
    # Número bins
    # =====================================
    n = len(valores)

    if bins == 'sturges':

        nbins = int(
            round(
                1 + 3.322 * np.log10(n)
            )
        )

    elif bins == 'sqrt':

        nbins = int(
            round(
                np.sqrt(n)
            )
        )

    elif bins == 'fd':

        q1 = valores.quantile(0.25)
        q3 = valores.quantile(0.75)

        iqr = q3 - q1

        amplitud = (
            2 * iqr
        ) / (n ** (1/3))

        nbins = int(
            round(
                (
                    valores.max()
                    - valores.min()
                ) / amplitud
            )
        )

    else:

        nbins = bins

    # =====================================
    # Estadísticos
    # =====================================
    asimetria = skew(valores)

    curt = kurtosis(
        valores,
        fisher=True
    )

    media = np.mean(valores)

    mediana = np.median(valores)

    # =====================================
    # Interpretación asimetría
    # =====================================
    if asimetria > 0:

        interp_asim = (
            'Asimetría positiva'
        )

    elif asimetria < 0:

        interp_asim = (
            'Asimetría negativa'
        )

    else:

        interp_asim = (
            'Distribución simétrica'
        )

    # =====================================
    # Interpretación curtosis
    # =====================================
    if curt > 0:

        interp_curt = (
            'Leptocúrtica'
        )

    elif curt < 0:

        interp_curt = (
            'Platicúrtica'
        )

    else:

        interp_curt = (
            'Mesocúrtica'
        )

    # =====================================
    # Histograma
    # =====================================
    fig = px.histogram(

        df,

        x=columna,

        nbins=nbins,

        marginal=marginal,

        opacity=0.80
    )

    # =====================================
    # Hover
    # =====================================
    fig.update_traces(

        marker_line_width=1.2,

        hovertemplate=
        '<b>Valor</b>: %{x}<br>' +
        '<b>Frecuencia</b>: %{y}<extra></extra>'
    )

    # =====================================
    # Líneas estadísticas
    # =====================================
    if mostrar_media:

        fig.add_vline(

            x=media,

            line_dash='dash',

            annotation_text='Media',

            annotation_position='top'
        )

    if mostrar_mediana:

        fig.add_vline(

            x=mediana,

            line_dash='dot',

            annotation_text='Mediana',

            annotation_position='bottom'
        )

    # =====================================
    # Subtítulo estadístico
    # =====================================
    subtitulo = ""

    if mostrar_asimetria:

        subtitulo += (
            f"Asimetría = {asimetria:.4f} "
            f"({interp_asim})"
        )

    if mostrar_curtosis:

        subtitulo += (
            f" | Curtosis = {curt:.4f} "
            f"({interp_curt})"
        )

    # =====================================
    # Layout
    # =====================================
    fig.update_layout(

        template='plotly_white',

        title={

            'text': (
                f'Histograma Analítico - '
                f'{columna}'
                f'<br><sup>{subtitulo}</sup>'
            ),

            'x': 0.5,

            'font': {
                'size': 22
            }
        },

        xaxis_title='Valores',

        yaxis_title='Densidad',

        width=width,

        height=height,

        bargap=0.05
    )

    fig.show()

    # =====================================
    # Retorno estadísticos
    # =====================================
    return {

        'media': media,

        'mediana': mediana,

        'asimetria': asimetria,

        'curtosis': curt,

        'interpretacion_asimetria': interp_asim,

        'interpretacion_curtosis': interp_curt
    }

def boxplot_variable_continua(
    df,
    columna,
    points='outliers',
    mostrar_media=True,
    color=None,
    orientation='v',
    width=700,
    height=550
):
    """
    Genera un boxplot interactivo para
    una variable continua.

    Parámetros
    ----------
    df : pandas.DataFrame

    columna : str
        Variable continua.

    points : str
        Mostrar puntos:
        - 'outliers'
        - 'all'
        - False

    mostrar_media : bool
        Mostrar media y desviación estándar.

    color : str
        Color cajas.

    orientation : str
        'v' -> vertical
        'h' -> horizontal

    width : int
        Ancho figura.

    height : int
        Alto figura.
    """

    # =====================================
    # Boxplot vertical
    # =====================================
    if orientation == 'v':

        fig = px.box(

            df,

            y=columna,

            points=points,

            color_discrete_sequence=(
                [color]
                if color is not None
                else None
            )
        )

        fig.update_layout(
            yaxis_title='Valores'
        )

        hover_template = (
            '<b>Valor</b>: %{y}<extra></extra>'
        )

    # =====================================
    # Boxplot horizontal
    # =====================================
    else:

        fig = px.box(

            df,

            x=columna,

            points=points,

            orientation='h',

            color_discrete_sequence=(
                [color]
                if color is not None
                else None
            )
        )

        fig.update_layout(
            xaxis_title='Valores'
        )

        hover_template = (
            '<b>Valor</b>: %{x}<extra></extra>'
        )

    # =====================================
    # Trazas
    # =====================================
    fig.update_traces(

        boxmean='sd'
        if mostrar_media
        else False,

        hovertemplate=hover_template
    )

    # =====================================
    # Layout
    # =====================================
    fig.update_layout(

        template='plotly_white',

        title={
            'text': (
                f'Diagrama de Caja - {columna}'
            ),

            'x': 0.5,

            'font': {
                'size': 22
            }
        },

        width=width,

        height=height
    )

    fig.show()

def poligono_frecuencias(
    df,
    columna,
    bins='sturges',
    mostrar_puntos=True,
    width=900,
    height=550
):
    """
    Genera un polígono de frecuencias
    para una variable continua.

    Parámetros
    ----------
    df : pandas.DataFrame

    columna : str
        Variable continua.

    bins : int o str
        Número de clases o método:
        - 'sturges'
        - 'sqrt'
        - 'fd'

    mostrar_puntos : bool
        Mostrar marcadores.

    width : int
        Ancho figura.

    height : int
        Alto figura.
    """

    # =====================================
    # Serie limpia
    # =====================================
    serie = df[columna].dropna()

    # =====================================
    # Número observaciones
    # =====================================
    n = len(serie)

    # =====================================
    # Número de bins
    # =====================================
    if bins == 'sturges':

        k = int(
            round(
                1 + 3.322 * np.log10(n)
            )
        )

    elif bins == 'sqrt':

        k = int(
            round(
                np.sqrt(n)
            )
        )

    elif bins == 'fd':

        q1 = serie.quantile(0.25)
        q3 = serie.quantile(0.75)

        iqr = q3 - q1

        amplitud = (
            2 * iqr
        ) / (n ** (1/3))

        k = int(
            round(
                (serie.max() - serie.min())
                / amplitud
            )
        )

    else:

        k = bins

    # =====================================
    # Intervalos
    # =====================================
    frecuencias, limites = np.histogram(
        serie,
        bins=k
    )

    # =====================================
    # Marcas de clase
    # =====================================
    marcas_clase = (
        limites[:-1]
        + limites[1:]
    ) / 2

    # =====================================
    # DataFrame
    # =====================================
    plot_df = pd.DataFrame({

        'Marca de Clase': marcas_clase,

        'Frecuencia': frecuencias
    })

    # =====================================
    # Polígono
    # =====================================
    fig = px.line(

        plot_df,

        x='Marca de Clase',

        y='Frecuencia',

        markers=mostrar_puntos
    )

    # =====================================
    # Hover
    # =====================================
    fig.update_traces(

        hovertemplate=
        '<b>Marca de Clase</b>: %{x}<br>' +
        '<b>Frecuencia</b>: %{y}<extra></extra>'
    )

    # =====================================
    # Layout
    # =====================================
    fig.update_layout(

        template='plotly_white',

        title={
            'text': (
                f'Polígono de Frecuencias - '
                f'{columna}'
            ),

            'x': 0.5,

            'font': {
                'size': 22
            }
        },

        xaxis_title='Marca de Clase',

        yaxis_title='Frecuencia',

        width=width,

        height=height
    )

    fig.show()

    return plot_df

def diagrama_dispersion(
    df,
    x,
    y,
    color=None,
    tendencia=True,
    metodo_corr='pearson',
    opacity=0.75,
    width=900,
    height=600
):
    """
    Genera un diagrama de dispersión interactivo
    entre dos variables cuantitativas.

    Parámetros
    ----------
    df : pandas.DataFrame

    x : str
        Variable eje X.

    y : str
        Variable eje Y.

    color : str
        Variable categórica opcional
        para colorear puntos.

    tendencia : bool
        Mostrar línea de tendencia.

    metodo_corr : str
        Método correlación:
        - 'pearson'
        - 'spearman'

    opacity : float
        Transparencia puntos.

    width : int
        Ancho figura.

    height : int
        Alto figura.
    """

    # =====================================
    # Datos limpios
    # =====================================
    datos = df[[x, y]].dropna()

    # =====================================
    # Correlación
    # =====================================
    if metodo_corr == 'pearson':

        corr, pvalor = pearsonr(
            datos[x],
            datos[y]
        )

    elif metodo_corr == 'spearman':

        corr, pvalor = spearmanr(
            datos[x],
            datos[y]
        )

    else:

        raise ValueError(
            "metodo_corr debe ser "
            "'pearson' o 'spearman'"
        )

    # =====================================
    # Interpretación
    # =====================================
    abs_corr = abs(corr)

    if abs_corr < 0.20:

        interpretacion = (
            'Correlación muy débil'
        )

    elif abs_corr < 0.40:

        interpretacion = (
            'Correlación débil'
        )

    elif abs_corr < 0.60:

        interpretacion = (
            'Correlación moderada'
        )

    elif abs_corr < 0.80:

        interpretacion = (
            'Correlación fuerte'
        )

    else:

        interpretacion = (
            'Correlación muy fuerte'
        )

    # =====================================
    # Scatterplot
    # =====================================
    fig = px.scatter(

        df,

        x=x,

        y=y,

        color=color,

        opacity=opacity,

        trendline='ols'
        if tendencia
        else None
    )

    # =====================================
    # Hover
    # =====================================
    fig.update_traces(

        hovertemplate=
        f'<b>{x}</b>: %{{x}}<br>' +
        f'<b>{y}</b>: %{{y}}<extra></extra>'
    )

    # =====================================
    # Layout
    # =====================================
    fig.update_layout(

        template='plotly_white',

        title={

            'text': (
                f'Diagrama de Dispersión'
                f'<br><sup>'
                f'Correlación ({metodo_corr}) = '
                f'{corr:.4f} | '
                f'{interpretacion}'
                f'</sup>'
            ),

            'x': 0.5,

            'font': {
                'size': 22
            }
        },

        xaxis_title=x,

        yaxis_title=y,

        width=width,

        height=height
    )

    fig.show()

    # =====================================
    # Covarianza
    # =====================================
    covarianza = np.cov(
        datos[x],
        datos[y]
    )[0, 1]

    # =====================================
    # Retorno
    # =====================================
    return {

        'correlacion': corr,

        'pvalor': pvalor,

        'covarianza': covarianza,

        'interpretacion': interpretacion
    }

def matriz_correlacion(
    df,
    variables=None,
    metodo='pearson',
    mostrar_texto=True,
    width=850,
    height=750
):
    """
    Genera una matriz de correlación interactiva.

    Parámetros
    ----------
    df : pandas.DataFrame

    variables : list
        Lista de variables numéricas.
        Si None toma todas las numéricas.

    metodo : str
        Método de correlación:
        - 'pearson'
        - 'spearman'
        - 'kendall'

    mostrar_texto : bool
        Mostrar coeficientes sobre mapa.

    width : int
        Ancho figura.

    height : int
        Alto figura.
    """

    # =====================================
    # Variables numéricas
    # =====================================
    if variables is None:

        datos = df.select_dtypes(
            include=np.number
        )

    else:

        datos = df[variables]

    # =====================================
    # Matriz correlación
    # =====================================
    corr = datos.corr(
        method=metodo
    )

    # =====================================
    # Heatmap
    # =====================================
    fig = px.imshow(

        corr,

        text_auto=
        '.2f'
        if mostrar_texto
        else False,

        aspect='auto',

        zmin=-1,
        zmax=1,

        color_continuous_scale='RdBu_r'
    )

    # =====================================
    # Hover
    # =====================================
    fig.update_traces(

        hovertemplate=
        '<b>X</b>: %{x}<br>' +
        '<b>Y</b>: %{y}<br>' +
        '<b>Correlación</b>: %{z:.4f}'
        '<extra></extra>'
    )

    # =====================================
    # Layout
    # =====================================
    fig.update_layout(

        template='plotly_white',

        title={

            'text': (
                f'Matriz de Correlación '
                f'({metodo.capitalize()})'
            ),

            'x': 0.5,

            'font': {
                'size': 24
            }
        },

        width=width,

        height=height,

        coloraxis_colorbar=dict(
            title='Correlación'
        )
    )

    fig.show()

    # =====================================
    # Retorno
    # =====================================
    return corr

def tabla_outliers(
    df,
    columna,
    metodo='iqr',
    mostrar_tabla=True,
    width=700,
    height=320
):
    """
    Genera una tabla vertical individual comparando
    métricas antes y después de eliminar outliers.
    """

    serie_original = df[columna].dropna()

    # =========================================
    # DETECCIÓN OUTLIERS
    # =========================================

    if metodo == 'iqr':

        Q1 = serie_original.quantile(0.25)
        Q3 = serie_original.quantile(0.75)

        IQR = Q3 - Q1

        limite_inferior = Q1 - 1.5 * IQR
        limite_superior = Q3 + 1.5 * IQR

        serie_filtrada = serie_original[
            (serie_original >= limite_inferior) &
            (serie_original <= limite_superior)
        ]

    elif metodo == 'zscore':

        media = serie_original.mean()
        desviacion = serie_original.std()

        z_scores = (
            (serie_original - media)
            / desviacion
        )

        serie_filtrada = serie_original[
            np.abs(z_scores) < 3
        ]

    else:

        raise ValueError(
            "metodo debe ser 'iqr' o 'zscore'"
        )

    # =========================================
    # TABLA VERTICAL
    # =========================================

    tabla = pd.DataFrame({

        'Métrica': [

            'Media',
            'Desviación estándar',
            'Asimetría',
            'Cantidad de datos',
            'Outliers eliminados'

        ],

        'Antes': [

            round(
                serie_original.mean(),
                4
            ),

            round(
                serie_original.std(),
                4
            ),

            round(
                skew(serie_original),
                4
            ),

            len(serie_original),

            '-'
        ],

        'Después': [

            round(
                serie_filtrada.mean(),
                4
            ),

            round(
                serie_filtrada.std(),
                4
            ),

            round(
                skew(serie_filtrada),
                4
            ),

            len(serie_filtrada),

            len(serie_original)
            - len(serie_filtrada)
        ]
    })

    # =========================================
    # TABLA VISUAL
    # =========================================

    if mostrar_tabla:

        fig = go.Figure(

            data=[

                go.Table(

                    header=dict(

                        values=[
                            f"<b>{col}</b>"
                            for col in tabla.columns
                        ],

                        fill_color='#1E3A8A',

                        font=dict(
                            color='white',
                            size=14
                        ),

                        align='center',

                        height=40
                    ),

                    cells=dict(

                        values=[
                            tabla[col]
                            for col in tabla.columns
                        ],

                        fill_color=[
                            [
                                '#F8FAFC'
                                if i % 2 == 0
                                else '#E2E8F0'
                                for i in range(len(tabla))
                            ]
                        ],

                        align='center',

                        font=dict(
                            color='#111827',
                            size=13
                        ),

                        height=34
                    )
                )
            ]
        )

        fig.update_layout(

            title={

                'text': (
                    f'Comparación Outliers - '
                    f'{columna}'
                ),

                'x': 0.5,

                'font': {
                    'size': 22
                }
            },

            width=width,

            height=height,

            margin=dict(
                l=20,
                r=20,
                t=70,
                b=20
            )
        )

        fig.show()

    return tabla

: 

In [ ]:
!pip install ydata_profiling scikit_posthocs -q

import os
import numpy as np
import pandas as pd
from pandas.plotting import table
from scipy import stats as st
from scipy.stats import kurtosis, pearsonr, skew, spearmanr
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import scikit_posthocs as sp
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from ydata_profiling import ProfileReport
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    TargetEncoder,
)

import joblib

In [ ]:
## Cargar el dataset
df_registraduria = pd.read_excel('/content/registraduria_database_clean.xlsx')

In [ ]:
# @title 3.3. Reporte General Inicial

profile = ProfileReport(
    df_registraduria,
    title="Reporte Calidad Datos"
)

profile.to_notebook_iframe()

## 3.4 Hallazgos del reporte inicial

XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXxxxxx

## 3.5. Limpieza de valores nulos

In [ ]:
## Eliminar valores de viaticos núlos
df_registraduria = df_registraduria[df_registraduria['total_travel_allowance'] != 0].copy()

## Nivel jerarquico de CONTRATISTAS y OTROS como 0
df_registraduria['position_level'] = df_registraduria['position_level'].fillna(0)

## 3.6. Limpieza outliers `distance_km`

In [ ]:
iqr_distance = stats.iqr(df_registraduria['distance_km'])
print(f"Rango intercuartílico para distance_km: {iqr_distance}")
q1_distance = df_registraduria['distance_km'].quantile(0.25)
q3_distance = df_registraduria['distance_km'].quantile(0.75)
distance_upper_bound = q3_distance + 1.5 * iqr_distance
distance_lower_bound = q1_distance - 1.5 * iqr_distance

n_antes = len(df_registraduria)

df_registraduria = df_registraduria[
    (df_registraduria['distance_km'] >= distance_lower_bound) &
    (df_registraduria['distance_km'] <= distance_upper_bound)
]

n_despues = len(df_registraduria)

print(f"Registros eliminados: {n_antes - n_despues} ({(n_antes - n_despues) / n_antes * 100:.2f}%)")
print(f"Registros restantes:  {n_despues}")

# 4.  Análisis Exploratorio de datos (EDA)

## 4.1. Análisis univariado de variables cualitativas.

### 4.1.1. Motivos de viaje

In [ ]:
grafico_barras_categorica(
    df=df_registraduria,
    columna='purpose',
    ordenar='freq',
)

grafico_pastel_categorica(
    df=df_registraduria,
    columna='purpose',
    ordenar='freq',
)

### 4.1.2. Posición del empleado

In [ ]:
grafico_barras_categorica(
    df=df_registraduria,
    columna='employee_position_norm',
    ordenar='freq',
)

grafico_pastel_categorica(
    df=df_registraduria,
    columna='employee_position_norm',
    ordenar='freq',
)

### 4.1.3. Ciudad de origen

In [ ]:
grafico_barras_categorica(
    df=df_registraduria,
    columna='city_origin',
    ordenar='freq',
)

# grafico_pastel_categorica(
#     df=df_registraduria,
#     columna='city_origin',
#     ordenar='index',
# )

### 4.1.4. Ciudad principal de destino

In [ ]:
grafico_barras_categorica(
    df=df_registraduria,
    columna='city_destination_main',
    ordenar='freq',
)

# grafico_pastel_categorica(
#     df=df_registraduria,
#     columna='city_destination_main',
#     ordenar='index',
# )

### 4.1.5. Es contratista

In [ ]:
contractor_labels = {0: 'No Contratista', 1 : 'Contratista'}
grafico_barras_categorica(
    df=df_registraduria,
    columna='is_contractor',
    ordenar='freq',
    labels= contractor_labels
)

grafico_pastel_categorica(
    df=df_registraduria,
    columna='is_contractor',
    ordenar='freq',
    labels= contractor_labels
)

## 4.2. Análisis univariado de variables cuantitativas.

### 4.2.1 Tablas de frecuencias para variables numéricas.</h2>

In [ ]:
tabla_frecuencias_continua(
    df=df_registraduria,
    columna='total_travel_allowance'
)

### 4.2.2. Histograma Analítico - `total_travel_allowance`

In [ ]:
histograma_variable_continua(
    df=df_registraduria,
    columna='total_travel_allowance')

### 4.2.3. Días de comision

In [ ]:
tabla_frecuencias_categorica(
    df=df_registraduria,
    columna='commission_days',
    ordenar='index',
    mostrar_acumuladas=True
)

### 4.2.4. Distancia en kilometros

In [ ]:
tabla_frecuencias_continua(
    df=df_registraduria,
    columna='distance_km'
)

### 4.2.5. Número de destinos


In [ ]:
tabla_frecuencias_categorica(
    df=df_registraduria,
    columna='destination_count',
    ordenar='index',
    mostrar_acumuladas=True
)

## 4.3. Análisis gráfico de distribuciones


### 4.3.1. Gastos totales de viaje

In [ ]:
print(f"Media: {round(np.mean(df_registraduria['total_travel_allowance']),4)}")

print(f"Mediana: {round(np.median(df_registraduria['total_travel_allowance']), 4)}")

desviacion_estandar = round(np.std(df_registraduria['total_travel_allowance']),4)
print(f"Desviación Estándar: {desviacion_estandar}")

coef_var = round((desviacion_estandar/np.mean(df_registraduria['total_travel_allowance']))*100, 4)
print(f"Coeficiente de Variación: {coef_var}")

num_outliers = contar_outliers_iqr(
    df_registraduria,
    'total_travel_allowance'
)

print(f"Número de outliers: {num_outliers}")

histograma_variable_continua(
    df=df_registraduria,
    columna='total_travel_allowance')

boxplot_variable_continua(
    df=df_registraduria,
    columna='total_travel_allowance')



### 4.3.2. Días comisionados

In [ ]:
print(f"Media: {round(np.mean(df_registraduria['commission_days']),4)}")

print(f"Mediana: {round(np.median(df_registraduria['commission_days']), 4)}")

desviacion_estandar = round(np.std(df_registraduria['commission_days']),4)
print(f"Desviación Estándar: {desviacion_estandar}")

coef_var = round((desviacion_estandar/np.mean(df_registraduria['commission_days']))*100, 4)
print(f"Coeficiente de Variación: {coef_var}")

num_outliers = contar_outliers_iqr(
    df_registraduria,
    'commission_days'
)

print(f"Número de outliers: {num_outliers}")

histograma_variable_continua(
    df=df_registraduria,
    columna='commission_days')

boxplot_variable_continua(
    df=df_registraduria,
    columna='commission_days')

### 4.3.3. Distancia en kilometros

In [ ]:
print(f"Media: {round(np.mean(df_registraduria['distance_km']),4)}")

print(f"Mediana: {round(np.median(df_registraduria['distance_km']), 4)}")

desviacion_estandar = round(np.std(df_registraduria['distance_km']),4)
print(f"Desviación Estándar: {desviacion_estandar}")

coef_var = round((desviacion_estandar/np.mean(df_registraduria['distance_km']))*100, 4)
print(f"Coeficiente de Variación: {coef_var}")

num_outliers = contar_outliers_iqr(
    df_registraduria,
    'distance_km'
)

print(f"Número de outliers: {num_outliers}")

histograma_variable_continua(
    df=df_registraduria,
    columna='distance_km')

boxplot_variable_continua(
    df=df_registraduria,
    columna='distance_km')

### 4.3.4. Número de destinos


In [ ]:
print(f"Media: {round(np.mean(df_registraduria['destination_count']),4)}")

print(f"Mediana: {round(np.median(df_registraduria['destination_count']), 4)}")

desviacion_estandar = round(np.std(df_registraduria['destination_count']),4)
print(f"Desviación Estándar: {desviacion_estandar}")

coef_var = round((desviacion_estandar/np.mean(df_registraduria['destination_count']))*100, 4)
print(f"Coeficiente de Variación: {coef_var}")

num_outliers = contar_outliers_iqr(
    df_registraduria,
    'destination_count'
)

print(f"Número de outliers: {num_outliers}")

histograma_variable_continua(
    df=df_registraduria,
    columna='destination_count')

boxplot_variable_continua(
    df=df_registraduria,
    columna='destination_count')

## 4.4. Asociación lineal entre variables

In [ ]:
corr = matriz_correlacion(
    df=df_registraduria,
    variables = ['total_travel_allowance', 'commission_days', 'position_level', 'distance_km', 'destination_count', 'is_contractor']
)


# 5. Pruebas de Hipotesis e Interpretación

## 5.1. Diferencias entre gastos de viaje para los diferentes cargos:

***¿El costo de los viáticos varía significativamente dependiendo del motivo de la comisión?***

- H0: Las medianas del costo de viáticos son iguales para todos cargos.
- H1: Al menos cargo tiiene una mediana de costos significativamente diferente a los demás.

In [ ]:
def shapiro_test(data, x, y, alpha=0.05):
    """
    x: variable categórica (grupos)
    y: variable continua
    """
    grupos = data[x].dropna().unique()

    for grupo in grupos:
        datos = data[data[x] == grupo][y].dropna()

        if len(datos) > 5000:
            datos = datos.sample(5000, random_state=42)

        W, p = stats.shapiro(datos)

        p_str = f"{p:.4e}" if p < 0.001 else f"{p:.4f}"

        print(f"📊 Variable categórica: {x} → {grupo}")
        print(f"📈 Variable continua: {y}")
        print(f"W = {W:.4f}")
        print(f"p-value = {p_str}")

        if p < alpha:
            print(f"❌ Los datos de {y.lower()} en {grupo} no siguen una distribución normal\n")
        else:
            print(f"✅ Los datos de {y.lower()} en {grupo} sí siguen una distribución normal\n")


In [ ]:
shapiro_test(df_registraduria,
             x = 'employee_position_norm',
             y = 'total_travel_allowance')

In [ ]:
def kruskal_test(data, x, y, alpha=0.05):
    """
    x: variable categórica
    y: variable continua
    """
    grupos = [
        data[data[x] == cat][y].dropna()
        for cat in data[x].dropna().unique()
    ]

    H, p = stats.kruskal(*grupos)

    p_str = f"{p:.4e}" if p < 0.001 else f"{p:.4f}"

    print(f"📊 Variable categórica: {x}")
    print(f"📈 Variable continua: {y}")
    print(f"H = {H:.4f}")
    print(f"p-value = {p_str}")

    if p < alpha:
        print("❌ Se rechaza H₀ → Hay diferencias significativas entre grupos\n")
    else:
        print("✅ No se rechaza H₀ → No hay evidencia de diferencias entre grupos\n")


In [ ]:
kruskal_test(
    df_registraduria,
    x='employee_position_norm',
    y='total_travel_allowance'
)

In [ ]:
kruskal_test(
    df_registraduria,
    x='position_level',
    y='total_travel_allowance'
)

In [ ]:
## Comparaciones post-hoc
resultado = sp.posthoc_dunn(df_registraduria, val_col='total_travel_allowance', group_col='employee_position_norm', p_adjust='bonferroni')
print(resultado)

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(resultado, dtype=bool))
labels = resultado.applymap(
    lambda v: "* " if v < 0.05 else ""
)
sns.heatmap(
    resultado,
    mask=mask,
    cmap="vlag",
    vmax=0.05,
    vmin=0,
    center=0.025,
    annot=labels,
    fmt="",
    linewidths=0.5,
    cbar_kws={"label": "p-valor (Truncado en 0.05)"},
)


plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.title(
    "Matriz de Valores p (Prueba de Dunn con ajuste de Holm)",
    fontsize=14,
    pad=20,
)

plt.tight_layout()
plt.show()


In [ ]:
## Comparaciones post-hoc
resultado = sp.posthoc_dunn(df_registraduria, val_col='total_travel_allowance', group_col='position_level', p_adjust='holm')
print(resultado)

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(resultado, dtype=bool))
labels = resultado.applymap(
    lambda v: "* " if v < 0.05 else ""
)
sns.heatmap(
    resultado,
    mask=mask,
    cmap="vlag",
    vmax=0.05,
    vmin=0,
    center=0.025,
    annot=labels,
    fmt="",
    linewidths=0.5,
    cbar_kws={"label": "p-valor (Truncado en 0.05)"},
)


plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.title(
    "Matriz de Valores p (Prueba de Dunn con ajuste de Holm)",
    fontsize=14,
    pad=20,
)

plt.tight_layout()
plt.show()

## 5.2. Diferencias de los costos de viaje según el proposito de viaje

***¿El costo de los viáticos varía significativamente dependiendo del motivo de la comisión?***

- H0: Las medianas del costo de viáticos son iguales para todos los propósitos de viaje.
- H1: Al menos un propósito de viaje tiene una mediana de costos significativamente diferente a los demás.

In [ ]:
shapiro_test(
    df_registraduria,
    x = 'purpose',
    y = 'total_travel_allowance'
)

In [ ]:
kruskal_test(
    df_registraduria,
    x='purpose',
    y='total_travel_allowance'
)

In [ ]:
## Comparaciones post-hoc
resultado = sp.posthoc_dunn(df_registraduria, val_col='total_travel_allowance', group_col='purpose', p_adjust='holm')
print(resultado)

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(resultado, dtype=bool))
labels = resultado.applymap(
    lambda v: "* " if v < 0.05 else ""
)
sns.heatmap(
    resultado,
    mask=mask,
    cmap="vlag",
    vmax=0.05,
    vmin=0,
    center=0.025,
    annot=labels,
    fmt="",
    linewidths=0.5,
    cbar_kws={"label": "p-valor (Truncado en 0.05)"},
)


plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.title(
    "Matriz de Valores p (Prueba de Dunn con ajuste de Holm)",
    fontsize=14,
    pad=20,
)

plt.tight_layout()
plt.show()

## 5.3. Diferencias entre el proposito de viaje y la duración del viaje

***¿La duración del viaje varía significativamente dependiendo del motivo de la comisión?***

- H0: Las medianas de la duración del viaje son iguales para todos los propósitos de viaje.
- H1: Al menos una duración de viaje tiene una mediana de costos significativamente diferente a los demás.


In [ ]:
shapiro_test(
    df_registraduria,
    x = 'purpose',
    y = 'commission_days'
)

In [ ]:
kruskal_test(
    df_registraduria,
    x='purpose',
    y='commission_days'
)

In [ ]:
## Comparaciones post-hoc
resultado = sp.posthoc_dunn(df_registraduria, val_col='commission_days', group_col='purpose', p_adjust='holm')
print(resultado)

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(resultado, dtype=bool))
labels = resultado.applymap(
    lambda v: "* " if v < 0.05 else ""
)
sns.heatmap(
    resultado,
    mask=mask,
    cmap="vlag",
    vmax=0.05,
    vmin=0,
    center=0.025,
    annot=labels,
    fmt="",
    linewidths=0.5,
    cbar_kws={"label": "p-valor (Truncado en 0.05)"},
)


plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.title(
    "Matriz de Valores p (Prueba de Dunn con ajuste de Holm)",
    fontsize=14,
    pad=20,
)

plt.tight_layout()
plt.show()

## 5.4. Diferencia del gasto entre contratistas y empleados

***¿Hay una diferencia significativa en el gasto de viaticos entre los empleados y los contratistas?***

- H0: No hay una diferencia en las medianas de gasto de viaje entre contratistas y empleados.
- H1: Hay una diferencia significativa en las mediandas de gasto de viaje entre los contratistas y empleados.

In [ ]:
shapiro_test(
    data=df_registraduria,
    x='is_contractor',
    y='total_travel_allowance'
  )

In [ ]:
from scipy import stats

def mann_whitney_test(data, x, y, alpha=0.05):
    """
    Realiza la prueba U de Mann-Whitney para dos grupos independientes.

    x: variable categórica (debe tener exactamente 2 grupos)
    y: variable continua
    """
    grupos = data[x].dropna().unique()


    if len(grupos) != 2:
        print(f"⚠️ Error: La prueba de Mann-Whitney requiere exactamente 2 grupos en '{x}'.")
        print(f"Grupos encontrados ({len(grupos)}): {grupos}\n")
        return

    grupo1, grupo2 = grupos[0], grupos[1]


    datos1 = data[data[x] == grupo1][y].dropna()
    datos2 = data[data[x] == grupo2][y].dropna()

    U, p = stats.mannwhitneyu(datos1, datos2, alternative='two-sided')

    mediana1 = datos1.median()
    mediana2 = datos2.median()

    p_str = f"{p:.4e}" if p < 0.001 else f"{p:.4f}"

    print(f"📊 Variable categórica: {x} ({grupo1} vs {grupo2})")
    print(f"📈 Variable continua: {y}")
    print(f"U = {U:.4f}")
    print(f"p-value = {p_str}")

    if p < alpha:
        print(f"❌ Hay diferencias significativas (p < {alpha}).")

        if mediana1 > mediana2:
            print(f"   ➡️ Los valores de {y.lower()} en '{grupo1}' son significativamente MAYORES que en '{grupo2}'.\n")
        elif mediana2 > mediana1:
            print(f"   ➡️ Los valores de {y.lower()} en '{grupo2}' son significativamente MAYORES que en '{grupo1}'.\n")
        else:
            # Caso raro donde las medianas son idénticas pero hay diferencia en las distribuciones
            print(f"   ➡️ Las distribuciones difieren, aunque sus medianas exactas son iguales.\n")
    else:
        print(f"✅ No hay diferencias significativas en {y.lower()} entre {grupo1} y {grupo2}\n")

In [ ]:
mann_whitney_test(
    data = df_registraduria,
    x='is_contractor',
    y='total_travel_allowance'
)

## 5.5. Correlación entre gasto y distancia:

***¿El aumento en la distancia del viaje (distance_km) se traduce de forma directamente proporcional en un aumento del costo del viático?***

- H0: No existe una correlación entre la distancia recorrida y el monto del viatico
- H1: Existe una correlación significativa entre la distancia y el costo.

In [ ]:
from scipy import stats

def pruebas_correlacion(data, x, y, alpha=0.05):
    """
    x: variable continua
    y: variable continua
    """
    df_clean = data[[x, y]].dropna()

    if len(df_clean) < 2:
        print("⚠️ Error: No hay suficientes datos válidos para calcular la correlación.\n")
        return

    var_x = df_clean[x]
    var_y = df_clean[y]

    def interpretar_relacion(coef):

        direccion = "POSITIVA ↗️" if coef > 0 else "NEGATIVA ↘️"


        abs_coef = abs(coef)
        if abs_coef >= 0.7:
            fuerza = "FUERTE"
        elif abs_coef >= 0.4:
            fuerza = "MODERADA "
        else:
            fuerza = "DÉBIL"

        return direccion, fuerza


    corr_p, p_p = stats.pearsonr(var_x, var_y)
    corr_s, p_s = stats.spearmanr(var_x, var_y)

    p_p_str = f"{p_p:.4e}" if p_p < 0.001 else f"{p_p:.4f}"
    p_s_str = f"{p_s:.4e}" if p_s < 0.001 else f"{p_s:.4f}"

    print(f"📊 Resultados de Correlación: '{x}' vs '{y}'\n")


    print("🔹 Pearson (Relación Lineal)")
    print(f"   Correlación: {corr_p:.4f}")
    print(f"   p-value: {p_p_str}")

    if p_p < alpha:
        dir_p, fue_p = interpretar_relacion(corr_p)
        print("   ✅ Se rechaza H₀: Existe relación lineal")
        print(f"   📌 Dirección: {dir_p}")
        print(f"   🔋 Fuerza: {fue_p}\n")
    else:
        print("   ❌ No se rechaza H₀: No hay evidencia de relación lineal\n")


    print("🔹 Spearman (Relación Monótona)")
    print(f"   Correlación: {corr_s:.4f}")
    print(f"   p-value: {p_s_str}")

    if p_s < alpha:
        dir_s, fue_s = interpretar_relacion(corr_s)
        print("   ✅ Se rechaza H₀: Existe relación monótona")
        print(f"   📌 Dirección: {dir_s}")
        print(f"   🔋 Fuerza: {fue_s}\n")
    else:
        print("   ❌ No se rechaza H₀: No hay evidencia de relación monótona\n")

In [ ]:
pruebas_correlacion(
    data = df_registraduria,
    x = 'distance_km',
    y = 'total_travel_allowance'
)

# 6. Construcción de Pipelines



In [ ]:
df_registraduria["full_date"]    = pd.to_datetime(df_registraduria["full_date"])
df_registraduria["month"]        = df_registraduria["full_date"].dt.month
df_registraduria["day_of_week"]  = df_registraduria["full_date"].dt.dayofweek
df_registraduria["quarter"]      = df_registraduria["full_date"].dt.quarter

NUM_REG = [
    "commission_days",
    "distance_km",
    "destination_count",
    "position_level"
]

CAT_OHE = [
    "purpose",
    "employee_position_norm",
    "month",
    "day_of_week",
]

CAT_TE = ["city_origin", "city_destination_main"]

FLAG_REG = [
    "is_contractor",
]

TARGET = "total_travel_allowance"

print("── Cardinalidad de variables categóricas ─────────────────────")
for c in CAT_REG:
    print(f"  {c:<30} {df_registraduria[c].nunique():>4} valores únicos")


prep_reg = ColumnTransformer(
    transformers=[
        ("num",  StandardScaler(),
                 NUM_REG),

        ("cat",  OneHotEncoder(
                     handle_unknown="ignore",
                     sparse_output=False,
                     drop="first",
                     min_frequency=10,          # Para disminuir la dimensionalidad de los destinos
                 ),
                 CAT_REG),

        ("flag", "passthrough",
                 FLAG_REG),
    ],
    remainder="drop",
)

pipeline_reg = Pipeline(
    steps=[
        ("preprocessor", prep_reg),
        ("regressor",    LinearRegression()),
    ]
)

In [ ]:
df_model = df_registraduria[df_registraduria[TARGET] > 0].copy()
print(f"\nRegistros: {len(df_registraduria):,} → {len(df_model):,} (se eliminan target=0)")

In [ ]:
df_model["log_target"] = np.log(df_model[TARGET])

print(f"\n  Target original  → media: {df_model[TARGET].mean()/1e6:.2f}M  "
      f"std: {df_model[TARGET].std()/1e6:.2f}M  "
      f"skew: {df_model[TARGET].skew():.2f}")
print(f"  log(target)      → media: {df_model['log_target'].mean():.2f}  "
      f"std: {df_model['log_target'].std():.2f}  "
      f"skew: {df_model['log_target'].skew():.2f}")


# 7. Entrenamiento de modelos

In [ ]:
ALL_FEATURES = NUM_REG + CAT_REG + FLAG_REG

X = df_model[ALL_FEATURES]
y = df_model["log_target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\n  Train : {len(X_train):,} registros")
print(f"  Test  : {len(X_test):,} registros")

In [ ]:
pipeline_reg

In [ ]:
X_encoded    = prep_reg.fit_transform(X_train)
feature_names = prep_reg.get_feature_names_out()
X_encoded_df  = pd.DataFrame(X_encoded, columns=feature_names)

print(f"\n  Dimensión tras OHE  : {X_encoded_df.shape}")
print(f"  Muestra de columnas : {list(feature_names[:5])} ... {list(feature_names[-3:])}")


In [ ]:
pipeline_reg.fit(X_train, y_train)
print("\n✓ Modelo entrenado")

# 8. Evaluación de modelos

## 8.1. Metricas de evaluación del modelo

In [ ]:
y_pred_log  = pipeline_reg.predict(X_test)
y_pred_orig = np.exp(y_pred_log)
y_test_orig = np.exp(y_test)

# Métricas en escala log (para evaluar el modelo directamente)
r2      = r2_score(y_test,      y_pred_log)
mae_log = mean_absolute_error(y_test, y_pred_log)

# Métricas en escala original
mae  = mean_absolute_error(y_test_orig,  y_pred_orig)
rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
mape = mean_absolute_percentage_error(y_test_orig, y_pred_orig) * 100

print("\n── Métricas en conjunto de prueba ─────────────────────────────────")
print(f"  R²  (escala log)    : {r2:.4f}")
print(f"  MAE (escala log)    : {mae_log:.4f}")
print(f"  MAE  (escala COP)   : {mae:>15,.0f}  COP")
print(f"  RMSE (escala COP)   : {rmse:>15,.0f}  COP")
print(f"  MAPE                : {mape:.2f}%")
print("────────────────────────────────────────────────────────────────────")

print(f"\n  → El modelo explica el {r2*100:.1f}% de la varianza del log-viático.")
print(f"  → Error medio de {mae/1e6:.2f}M COP (~{mape:.0f}% del valor real).")

## 8.2. Scatter plot e histograma

In [ ]:
df_resultados = pd.DataFrame({
    "y_real": y_test_orig,
    "y_pred": y_pred_orig
})

r, p = pearsonr(
    df_resultados["y_real"],
    df_resultados["y_pred"]
)

# SCATTER INTERACTIVO
fig = px.scatter(
    df_resultados,
    x="y_real",
    y="y_pred",
    opacity=0.70,
    trendline="ols",
    trendline_color_override="red"
)

# LÍNEA IDEAL
min_val = min(
    df_resultados["y_real"].min(),
    df_resultados["y_pred"].min()
)

max_val = max(
    df_resultados["y_real"].max(),
    df_resultados["y_pred"].max()
)

fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode='lines',
        name='Predicción Perfecta',
        line=dict(
            dash='dash',
            width=3,
            color='black'
        ),
        hovertemplate='<b>Línea Ideal</b><extra></extra>'
    )
)


fig.update_layout(
    template='plotly_white',
    title={
        'text': 'Viáticos Reales vs. Predicciones del Modelo' +
        f'<br><sup>Correlación Pearson = {r:.4f} | p-valor = {p:.6f}</sup>',
        'x': 0.5,
        'font': {'size': 24}
    },
    xaxis_title='Viático Real Pagado (COP)',
    yaxis_title='Predicción del Modelo (COP)',
    width=1000,
    height=750,
    legend_title='Referencia'
)

fig.update_traces(
    selector=dict(mode='markers'),
    marker=dict(
        size=8,
        line=dict(width=1, color='DarkSlateGrey')
    ),
    hovertemplate=
    '<b>Valor Real</b>: $%{x:,.0f} COP<br>' +
    '<b>Predicción</b>: $%{y:,.0f} COP<extra></extra>'
)

fig.show()

In [ ]:
from scipy.stats import skew,kurtosis
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import skew, kurtosis

df_resultados["error"] = df_resultados["y_real"] - df_resultados["y_pred"]
media_error = df_resultados["error"].mean()
std_error = df_resultados["error"].std()
asimetria = skew(df_resultados["error"].dropna())
curtosis_error = kurtosis(df_resultados["error"].dropna())


fig = px.histogram(
    df_resultados,
    x="error",
    nbins=30,
    marginal="box",
    opacity=0.80
)

# LÍNEA MEDIA
fig.add_vline(
    x=media_error,
    line_width=3,
    line_dash="dash",
    annotation_text=f"Media = {media_error:,.0f} COP",
    annotation_position="top right"
)

# HOVER (Actualizado para mostrar formato moneda)
fig.update_traces(
    hovertemplate=
    '<b>Error</b>: $%{x:,.0f} COP<br>' +
    '<b>Frecuencia</b>: %{y}<extra></extra>',
    selector=dict(type='histogram') # Asegura que solo aplique al histograma y no al boxplot superior
)

# =====================================#
# 4. LAYOUT
# =====================================#
fig.update_layout(
    template='plotly_white',
    title={
        'text': 'Distribución de Errores (Residuos)' +
        f'<br><sup>Media = {media_error:,.0f} COP | Desv = {std_error:,.0f} COP | ' +
        f'Asimetría = {asimetria:.2f} | Curtosis = {curtosis_error:.2f}</sup>',
        'x': 0.5,
        'font': {'size': 22}
    },
    xaxis_title='Error (COP)',
    yaxis_title='Frecuencia',
    width=1100,
    height=700,
    bargap=0.05
)

fig.show()

## 8.3. Validación cruzada

In [ ]:
cv_results = cross_validate(
    pipeline_reg, X, y,
    cv=10,
    scoring=["r2", "neg_mean_absolute_error"],
    return_train_score=True,
)

train_r2 = cv_results["train_r2"]
test_r2  = cv_results["test_r2"]

print("\n── Cross-Validation (10-fold) ──────────────────────────────────────")
print(f"  R² train      : {train_r2.mean():.4f}  ±  {train_r2.std():.4f}")
print(f"  R² validación : {test_r2.mean():.4f}  ±  {test_r2.std():.4f}")
print(f"  Overfitting   : {train_r2.mean() - test_r2.mean():.4f}")
print(f"  R² por fold   : {[round(v,4) for v in test_r2]}")
print("────────────────────────────────────────────────────────────────────")



In [ ]:
def predecir_viatico(commission_days: float, distance_km: float,
                     destination_count: int, purpose: str,
                     employee_position_norm: str, city_origin: str,
                     city_destination_main: str, month: int, position_level: int,
                     day_of_week: int, is_contractor: int = 0) -> dict:
    """
    Predice el monto del viático en pesos colombianos.

    Retorna: dict con prediccion_cop, ic_inf_cop, ic_sup_cop
    """
    entrada = pd.DataFrame([{
        "commission_days":      commission_days,
        "distance_km":          distance_km,
        "destination_count":    destination_count,
        "purpose":              purpose,
        "employee_position_norm": employee_position_norm,
        "city_origin":          city_origin,
        "city_destination_main": city_destination_main,
        "month":                month,
        "day_of_week":          day_of_week,
        "is_contractor":        is_contractor,
        "position_level":       position_level
    }])

    log_pred   = pipeline_reg.predict(entrada)[0]
    sigma_log  = np.std(resid)          # error estándar de residuos de test

    return {
        "prediccion_cop" : int(np.exp(log_pred)),
        "ic_inf_cop"     : int(np.exp(log_pred - 1.96 * sigma_log)),
        "ic_sup_cop"     : int(np.exp(log_pred + 1.96 * sigma_log)),
        "log_prediccion" : round(log_pred, 4),
    }

# Ejemplo
ej = predecir_viatico(
    commission_days=3, distance_km=350, destination_count=1,
    purpose="Visitas de Supervisión Contractual",
    employee_position_norm="PROFESIONAL",
    city_origin="BOGOTA D.C.", city_destination_main="MEDELLIN",
    month=6, day_of_week=1, is_contractor=0, position_level=1
)
print(f"\n── Ejemplo de predicción ────────────────────────────────────────")
print(f"  Predicción  : {ej['prediccion_cop']:>12,.0f} COP")
print(f"  IC 95%      : [{ej['ic_inf_cop']:,.0f}  —  {ej['ic_sup_cop']:,.0f}] COP")

In [ ]:
# import joblib
# joblib.dump(pipeline_reg, "modelo_viaticos_v1.pkl")
# modelo = joblib.load("modelo_viaticos_v1.pkl")
# pred   = modelo.predict(nueva_fila_df)

In [ ]:
def evaluar_modelo(nombre, pipeline, X_te, y_te, X_full, y_full):
    y_pred_log  = pipeline.predict(X_te)
    y_pred_orig = np.exp(y_pred_log)
    y_te_orig   = np.exp(y_te)

    r2   = r2_score(y_te,     y_pred_log)
    mae  = mean_absolute_error(y_te_orig,  y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_te_orig, y_pred_orig))
    mape = mean_absolute_percentage_error(y_te_orig, y_pred_orig) * 100

    cv = cross_validate(pipeline, X_full, y_full, cv=5,
                        scoring="r2", return_train_score=True)
    cv_val   = cv["test_score"].mean()
    cv_std   = cv["test_score"].std()
    cv_train = cv["train_score"].mean()

    print(f"── {nombre} ────────────────────────────────────────")
    print(f"  R²   (test)  : {r2:.4f}")
    print(f"  CV R² (val.) : {cv_val:.4f}  ±  {cv_std:.4f}")
    print(f"  Overfitting  : {cv_train - cv_val:.4f}")
    print(f"  MAE          : {mae:>12,.0f} COP  ({mae/1e6:.3f} M)")
    print(f"  RMSE         : {rmse:>12,.0f} COP")
    print(f"  MAPE         : {mape:.2f}%\n")

    return {
        "nombre": nombre, "r2": r2, "mae": mae, "rmse": rmse, "mape": mape,
        "cv_val": cv_val, "cv_std": cv_std, "cv_train": cv_train,
        "ovf": cv_train - cv_val,
        "y_pred_log": y_pred_log, "y_pred_orig": y_pred_orig,
        "y_te_orig":  y_te_orig,  "resid": y_te.values - y_pred_log,
    }

## 8.4. Usando árboles de decisión HistGradientBoosting

In [ ]:
NUM_HGB  = ["commission_days", "distance_km", "destination_count", "position_level"]
CAT_ORD  = ["purpose", "employee_position_norm", "month",
            "day_of_week", "quarter", "city_origin"]        # ≤255 cats → OrdinalEnc
CAT_TE4  = ["city_destination_main"]                        # >255 cats → TargetEnc
FLAG_HGB = ["is_contractor"]
ALL_HGB  = NUM_HGB + FLAG_HGB + CAT_TE4 + CAT_ORD

Xh = df_model[ALL_HGB]
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, y, test_size=0.2, random_state=42)

# Índices de columnas categóricas (OrdinalEncoded) para que HGB las trate bien
n_num_flag_te = len(NUM_HGB) + len(FLAG_HGB) + len(CAT_TE4)   # columnas antes de ords
cat_idx_hgb   = list(range(n_num_flag_te, n_num_flag_te + len(CAT_ORD)))

prep4 = ColumnTransformer([
    ("num",  StandardScaler(),  NUM_HGB),
    ("flag", "passthrough",    FLAG_HGB),
    ("te",   TargetEncoder(target_type="continuous", smooth="auto"), CAT_TE4),
    ("ord",  OrdinalEncoder(handle_unknown="use_encoded_value",
                            unknown_value=-1), CAT_ORD),
], remainder="drop")

pipe4 = Pipeline([
    ("pre", prep4),
    ("reg", HistGradientBoostingRegressor(
        max_iter        = 400,
        learning_rate   = 0.05,
        max_depth       = 6,
        min_samples_leaf= 20,
        l2_regularization = 0.1,
        categorical_features = cat_idx_hgb,
        random_state    = 42,
    )),
])

pipe4.fit(Xh_tr, yh_tr)
r4m = evaluar_modelo("HistGradientBoosting", pipe4, Xh_te, yh_te, Xh, y)


In [ ]:
_sigma_log = np.std(r4m["resid"])   # error estándar del modelo en test

def predecir_viatico_hgb(commission_days, distance_km, destination_count,
                          position_level, city_origin, city_destination_main,
                          purpose, employee_position_norm,
                          month, day_of_week, quarter,
                          is_contractor=0):
    """
    Predice el viático en COP usando el modelo HGB entrenado.
    Devuelve predicción puntual e intervalo de confianza del 95%.
    """
    entrada = pd.DataFrame([{
        "commission_days":       commission_days,
        "distance_km":           distance_km,
        "destination_count":     destination_count,
        "position_level":        position_level,
        "is_contractor":         is_contractor,
        "city_destination_main": city_destination_main,
        "purpose":               purpose,
        "employee_position_norm":employee_position_norm,
        "month":                 month,
        "day_of_week":           day_of_week,
        "quarter":               quarter,
        "city_origin":           city_origin,
    }])
    log_pred = pipe4.predict(entrada)[0]
    return {
        "prediccion_cop": int(np.exp(log_pred)),
        "ic_inf_cop":     int(np.exp(log_pred - 1.96 * _sigma_log)),
        "ic_sup_cop":     int(np.exp(log_pred + 1.96 * _sigma_log)),
    }

ej = predecir_viatico_hgb(
    commission_days=3, distance_km=350, destination_count=1,
    purpose="Visitas de Supervisión Contractual",
    employee_position_norm="PROFESIONAL",
    city_origin="BOGOTA D.C.", city_destination_main="MEDELLIN",
    month=6, day_of_week=1, is_contractor=0, position_level=1, quarter = 1
)
print(f"\nEjemplo de predicción HGB:")
print(f"  Predicción : {ej['prediccion_cop']:>12,.0f} COP")
print(f"  IC 95%     : [{ej['ic_inf_cop']:,.0f}  —  {ej['ic_sup_cop']:,.0f}] COP")

# 9. Conclusiones